# Mean Absolute Error (MAE) — Beginner Masterclass

**Reading:** [GeeksforGeeks — MAE in PyTorch](https://www.geeksforgeeks.org/python/how-to-measure-the-mean-absolute-error-mae-in-pytorch/) and [Medium — Mean Absolute Error](https://medium.com/@20__80__/mean-absolute-error-mae-machine-learning-ml-b9b4afc63077)

This notebook teaches MAE with manual calculations, plain Python, NumPy, PyTorch, TensorFlow/Keras, visual comparisons, and a small regression experiment.

## Learning goals
- Calculate residuals, absolute errors, and MAE by hand.
- Understand why MAE is less sensitive to outliers than MSE.
- Use `torch.nn.L1Loss` and Keras `MeanAbsoluteError`.
- Understand the nondifferentiable point at zero.
- Train a small regression model with MAE.

## 1. Formula and intuition

Mean Absolute Error is the average absolute difference between predictions and true values:

\[
\boxed{\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|}
\]

The steps are: calculate actual-minus-predicted error, take its absolute value, sum all absolute errors, and divide by \(n\).

MAE is common for regression. Unlike MSE, it does not square the error, so its penalty grows linearly. MAE is often less sensitive to outliers than MSE and is measured in the target's original units.

## 2. Worked example

Actual: \(y=[3,-0.5,2,7]\)

Predicted: \(\hat y=[2.5,0,2,8]\)

Absolute errors:

\[
[0.5,0.5,0,1]
\]

\[
\operatorname{MAE}=\frac{0.5+0.5+0+1}{4}=\boxed{0.5}.
\]

In [ ]:
# Plain Python only
y_true = [3.0, -0.5, 2.0, 7.0]
y_pred = [2.5, 0.0, 2.0, 8.0]
errors = [a - p for a, p in zip(y_true, y_pred)]
absolute_errors = [abs(e) for e in errors]
mae_manual = sum(absolute_errors) / len(absolute_errors)
print("Errors:", errors)
print("Absolute errors:", absolute_errors)
print("Sum absolute error:", sum(absolute_errors))
print("Number of values:", len(absolute_errors))
print("Manual MAE:", mae_manual)

## 3. NumPy implementation

In [ ]:
import numpy as np
actual = np.array(y_true, dtype=np.float64)
predicted = np.array(y_pred, dtype=np.float64)
abs_errors = np.abs(actual - predicted)
mae_numpy = abs_errors.mean()
print("Absolute errors:", abs_errors)
print("MAE:", mae_numpy)
assert np.isclose(mae_numpy, 0.5)

## 4. PyTorch `L1Loss`

In [ ]:
import torch
import torch.nn as nn
yt = torch.tensor(y_true, dtype=torch.float32)
yp = torch.tensor(y_pred, dtype=torch.float32)
print("PyTorch MAE:", nn.L1Loss()(yp, yt).item())
print("Elementwise absolute errors:", nn.L1Loss(reduction="none")(yp, yt))
print("Sum absolute error:", nn.L1Loss(reduction="sum")(yp, yt).item())

## 5. TensorFlow / Keras implementation

In [ ]:
import tensorflow as tf
yt_tf = tf.constant(y_true, dtype=tf.float32)
yp_tf = tf.constant(y_pred, dtype=tf.float32)
mae_tf = tf.keras.losses.MeanAbsoluteError()(yt_tf, yp_tf)
print("Keras MAE:", float(mae_tf.numpy()))
assert np.isclose(float(mae_tf.numpy()), 0.5, atol=1e-6)

## 6. Why MAE is less sensitive to outliers than MSE

With an error magnitude of 10, MAE contributes 10 while MSE contributes 100. Both increase due to an outlier, but MSE penalizes large error magnitudes quadratically.

MAE can be easier to interpret because its value uses the same unit as the target. It is not completely immune to outliers.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
residuals = np.linspace(-10, 10, 401)
plt.figure(figsize=(8,4))
plt.plot(residuals, np.abs(residuals), label="Absolute error (MAE term)")
plt.plot(residuals, residuals**2, label="Squared error (MSE term)")
plt.xlabel("Residual"); plt.ylabel("Per-element penalty")
plt.title("Absolute vs squared error")
plt.legend(); plt.grid(alpha=.25); plt.show()

def mae(a,p): return np.mean(np.abs(np.array(a)-np.array(p)))
def mse(a,p): return np.mean((np.array(a)-np.array(p))**2)
y = [10,20,30,40,50]
pred_small = [11,19,29,41,51]
pred_outlier = [11,19,29,41,100]
display(pd.DataFrame({
    "MAE": [mae(y,pred_small), mae(y,pred_outlier)],
    "MSE": [mse(y,pred_small), mse(y,pred_outlier)]
}, index=["Small errors", "One large error"]))

## 7. Derivative and the point at zero

For one sample, \(L=|\hat y-y|\). Its derivative with respect to \(\hat y\) is \(-1\) below the target and \(+1\) above the target (using \(\hat y-y\) as the residual convention). At exactly zero error the absolute value function is not differentiable in the classical sense; optimizers use a valid subgradient convention, commonly zero at the kink.

For a mean over \(n\) elements, the elementwise gradient is scaled by \(1/n\).

In [ ]:
# PyTorch shows the subgradient used by its autograd implementation.
pred = torch.tensor([2.0, 3.0, 4.0], requires_grad=True)
target = torch.tensor([3.0, 3.0, 2.0])
loss = nn.L1Loss()(pred, target)
loss.backward()
print("MAE:", loss.item())
print("Predictions:", pred.detach().numpy())
print("Targets:", target.numpy())
print("Gradient wrt predictions:", pred.grad.numpy())

## 8. Train a regression model on a dataset with outliers

We generate a target around \(y=2.5x-0.7\), then deliberately add a few large label errors. Two identical linear models are trained separately, one with MAE and one with MSE. This demonstrates how each loss responds to outliers. The curves measure different objective functions, so compare their trends rather than their absolute numeric heights.

In [ ]:
import numpy as np, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

rng = np.random.default_rng(9)
torch.manual_seed(9)
X = rng.uniform(-2, 2, (800,1)).astype("float32")
Y = (2.5*X - .7 + rng.normal(0,.25,(800,1))).astype("float32")
outlier_rng = np.random.default_rng(99)
idx = outlier_rng.choice(len(Y), size=40, replace=False)
Y[idx] += outlier_rng.choice([-1,1], size=(40,1)).astype("float32") * 8.0
loader = DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)), batch_size=32, shuffle=True)

def train(loss_name, epochs=60):
    torch.manual_seed(15)
    model = nn.Linear(1,1)
    opt = torch.optim.Adam(model.parameters(), lr=.02)
    criterion = nn.L1Loss() if loss_name == "MAE" else nn.MSELoss()
    history = []
    for _ in range(epochs):
        total, n = 0., 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb); n += len(xb)
        history.append(total/n)
    return model, history

model_mae, hist_mae = train("MAE")
model_mse, hist_mse = train("MSE")
print("MAE fit slope/bias:", model_mae.weight.item(), model_mae.bias.item())
print("MSE fit slope/bias:", model_mse.weight.item(), model_mse.bias.item())

plt.figure(figsize=(8,4))
plt.plot(hist_mae, label="MAE objective")
plt.plot(hist_mse, label="MSE objective")
plt.xlabel("Epoch"); plt.ylabel("Training loss (different scales)")
plt.title("Regression training with outlier labels")
plt.legend(); plt.grid(alpha=.25); plt.show()

## 9. Common mistakes and debugging

- MAE is typically a regression loss/metric, not the standard loss for ordinary classification.
- PyTorch `L1Loss` inputs and targets should have the same shape.
- `"mean"`, `"sum"`, and `"none"` have different reduction behavior.
- MAE is less sensitive to outliers than MSE, not completely insensitive.
- The absolute value is nondifferentiable at zero; framework autograd handles this using a subgradient convention.
- A metric can be MAE even when a different loss is optimized during training.

## 10. Quiz

1. Actual `[3,7,10]`, prediction `[2,9,10]`: MAE?
2. What does `torch.nn.L1Loss()` return by default?
3. Why is MAE less sensitive than MSE to a large error?
4. Does MAE have the same unit as the target?

**Answers:** 1. \((1+2+0)/3=1\). 2. Mean of absolute elementwise errors. 3. It grows linearly, not quadratically. 4. Yes.

## Sources and official documentation

### All six provided reading links
- [GeeksforGeeks — Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)
- [GeeksforGeeks — Mean Squared Error](https://www.geeksforgeeks.org/maths/mean-squared-error/)
- [GeeksforGeeks — MAE in PyTorch](https://www.geeksforgeeks.org/python/how-to-measure-the-mean-absolute-error-mae-in-pytorch/)
- [Medium — Mean Absolute Error](https://medium.com/@20__80__/mean-absolute-error-mae-machine-learning-ml-b9b4afc63077)
- [GeeksforGeeks — Binary Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/binary-cross-entropy-log-loss-for-binary-classification/)
- [GeeksforGeeks — Categorical Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/categorical-cross-entropy-in-multi-class-classification/)

### Official API docs
- [PyTorch L1Loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.L1Loss.html)
- [Keras regression losses](https://keras.io/api/losses/regression_losses/)
- [PyTorch autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

The notebook summarizes the sources in original wording rather than copying the full articles. If an error occurs, check shapes, dtypes, reduction settings, and the official API documentation.